In [ ]:
# -*- coding: utf-8 -*-
"""Untitled1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1QCwBGuGrP6zXQwTYnFdxMQfkUdga5I7G
"""

print('Installing essential packages...')
!pip install -q \
    accelerate \
    peft \
    torchaudio \
    datasets \
    jiwer \
    gradio \
    tqdm \
    evaluate \
    bitsandbytes>=0.46.1 \
    transformers>=4.38.0

print('Installation complete. Please restart the runtime now (Runtime > Restart runtime...).')

import torch
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    BitsAndBytesConfig,
)
from peft import prepare_model_for_kbit_training

# ─────────────────────────────────────────────────────────────
# CONSTANTS — imported by all later phases
# ─────────────────────────────────────────────────────────────
MODEL_NAME = "openai/whisper-small"
LANGUAGE   = "en"
TASK       = "transcribe"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")
print("-" * 50)

USE_8BIT = True if DEVICE == "cuda" else False

if USE_8BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,         # outlier threshold — default is fine
        llm_int8_has_fp16_weight=False, # store weights as int8, not fp16
    )
    print("Step 1b: BitsAndBytes 8-bit config created.")
else:
    bnb_config = None
    print("Step 1b: Skipping quantization (CPU mode — will use float32).")

print("Step 1c: Loading Whisper-small model weights...")

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto" if DEVICE == "cuda" else None,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)

if DEVICE == "cpu":
    model = model.to(DEVICE)

# ── FIX 1: Clear forced_decoder_ids ──────────────────────────
# Whisper normally forces <|en|><|transcribe|> tokens at positions 1 & 2
# of every generated sequence. During TRAINING this causes a mismatch:
# the forced tokens get inserted into the decoder inputs but NOT into
# the labels, so cross-entropy loss is computed on misaligned sequences.
# Setting to None disables forcing — the model learns from labels alone.
model.config.forced_decoder_ids = None

# ── FIX 2: Clear suppress_tokens ─────────────────────────────
# Whisper suppresses certain tokens (e.g. blank, padding) during beam
# search. During training this modifies logits before loss computation,
# silently zeroing out gradients for those token positions.
model.config.suppress_tokens = []

print(f"Step 1c: Model loaded on {DEVICE}.")
print(f"  forced_decoder_ids : {model.config.forced_decoder_ids}  (should be None)")
print(f"  suppress_tokens    : {model.config.suppress_tokens}     (should be [])")

if USE_8BIT:
    # ── FIX 3: prepare_model_for_kbit_training ────────────────
    # Must be called AFTER from_pretrained() and BEFORE LoRA injection.
    # Skipping this = LoRA adapters initialize but receive zero gradients.
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,  # saves VRAM during backprop
    )
    print("Step 1d: Model prepared for k-bit (8-bit) LoRA training.")
    print("  - LayerNorms cast to float32")
    print("  - Gradient checkpointing enabled")
else:
    print("Step 1d: Skipping kbit preparation (CPU mode).")

print("Step 1e: Loading WhisperProcessor...")

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task=TASK,
)

print("Step 1e: Processor loaded.")
print(f"  Feature extractor sampling rate : {processor.feature_extractor.sampling_rate} Hz")
print(f"  Tokenizer vocab size            : {processor.tokenizer.vocab_size}")
print(f"  Language token                  : {processor.tokenizer.language}")
print(f"  Task token                      : {processor.tokenizer.task}")

print("── Sanity Check 1: Feature extractor shape ──")

dummy_audio = torch.zeros(16000 * 5)  # 5 seconds of silence at 16 kHz

input_features = processor.feature_extractor(
    dummy_audio.numpy(),
    sampling_rate=16000,
    return_tensors="pt",
).input_features

print(f"  Dummy audio length   : {dummy_audio.shape[0]} samples (5 sec)")
print(f"  input_features shape : {input_features.shape}")
assert input_features.shape == torch.Size([1, 80, 3000]), \
    f"Expected [1, 80, 3000], got {input_features.shape}"
print("  ✓ Shape correct: [1, 80, 3000]")

print()
print("── Sanity Check 2: Tokenizer (no BOS token in output) ──")

# With add_special_tokens=False the BOS/SOT token is excluded.
# This is how we will tokenize in Phase 2 — important to verify here.
sample_text   = "hello world this is a lyrics test"
with_bos    = processor.tokenizer(sample_text).input_ids
without_bos = processor.tokenizer(sample_text, add_special_tokens=False).input_ids

print(f"  With    add_special_tokens (default) : {with_bos}")
print(f"  Without add_special_tokens           : {without_bos}")
print(f"  BOS token ID removed: {with_bos[0]} (first element gone in labels version)")

print()
print("── Sanity Check 3: Parameter counts ──")

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total model parameters     : {total_params / 1e6:.1f}M")
print(f"  Trainable parameters       : {trainable_params / 1e6:.1f}M")
print(f"  Trainable %                : {100 * trainable_params / total_params:.1f}%")
print()
print("  NOTE: After LoRA injection in Phase 3, trainable % will drop to ~1-2%.")

print()
print("Phase 1 complete ✓  Ready for Phase 2 (Data Preparation).")

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torchaudio
import torchaudio.transforms as T
import torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset as TorchDataset, DataLoader

# ── Import from Phase 1 ───────────────────────────────────────
# If running in the same Colab session as Phase 1, processor and
# DEVICE are already in memory. Otherwise the fallback re-loads them.
try:
    from step1_setup import processor, DEVICE
except ImportError:
    from transformers import WhisperProcessor
    processor = WhisperProcessor.from_pretrained(
        "openai/whisper-small", language="en", task="transcribe"
    )
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device  : {DEVICE}")
print(f"Processor     : {processor.__class__.__name__}")

from google.colab import drive
drive.mount('/content/drive')

NUS_ROOT     = "/content/drive/MyDrive/datasets/NUS-48E"
JAMENDO_ROOT = "/content/drive/MyDrive/datasets/jamendo"
OUTPUT_DIR   = "/content/processed"

TARGET_SR      = 16000   # Hz — Whisper's required sample rate
MAX_DURATION   = 30.0    # seconds — Whisper's hard encoder limit
MIN_DURATION   = 1.0     # seconds — skip clips shorter than this
CHUNK_OVERLAP  = 5.0     # seconds — overlap between Jamendo chunks

TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15
# TEST_RATIO   = 0.15 (implicit: whatever remains)

# ── FIX: seed ALL random sources for full reproducibility ─────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"NUS root       : {NUS_ROOT}")
print(f"Jamendo root   : {JAMENDO_ROOT}")
print(f"Target SR      : {TARGET_SR} Hz")
print(f"Max duration   : {MAX_DURATION}s")
print(f"Chunk overlap  : {CHUNK_OVERLAP}s")
print(f"Output dir     : {OUTPUT_DIR}")
print("-" * 50)

def load_and_resample(audio_path: str) -> torch.Tensor | None:
    """
    Load audio and resample to TARGET_SR.
    Returns 1D float32 tensor or None if the file is corrupt.
    """
    try:
        waveform, sr = torchaudio.load(audio_path)          # [C, T]

        # Stereo → mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)   # [1, T]

        # Resample if needed
        if sr != TARGET_SR:
            resampler = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
            waveform  = resampler(waveform)

        return waveform.squeeze(0)  # [T] — 1D

    except Exception as e:
        print(f"  [ERROR] Could not load {audio_path}: {e}")
        return None

def chunk_audio(
    waveform:    torch.Tensor,
    chunk_sec:   float = MAX_DURATION,
    overlap_sec: float = CHUNK_OVERLAP,
) -> list[torch.Tensor]:
    """
    Split a long waveform into overlapping chunks of `chunk_sec` seconds.
    The final chunk is kept if it meets MIN_DURATION.

    Args:
        waveform   : 1D tensor at TARGET_SR
        chunk_sec  : length of each chunk in seconds  (default: 30)
        overlap_sec: overlap between consecutive chunks (default: 5)

    Returns:
        List of 1D tensors, each at most chunk_sec * TARGET_SR samples.
    """
    chunk_size  = int(chunk_sec   * TARGET_SR)
    stride_size = int((chunk_sec - overlap_sec) * TARGET_SR)
    min_size    = int(MIN_DURATION * TARGET_SR)
    chunks      = []
    start       = 0

    while start + chunk_size <= len(waveform):
        chunks.append(waveform[start : start + chunk_size])
        start += stride_size

    # Keep the final short remainder if long enough
    remainder = waveform[start:]
    if len(remainder) >= min_size:
        chunks.append(remainder)

    return chunks


def pad_or_trim(waveform: torch.Tensor) -> torch.Tensor:
    """
    Trim to MAX_DURATION for NUS-48E samples (which are already short phrases).
    """
    max_samples = int(MAX_DURATION * TARGET_SR)
    return waveform[:max_samples]

def clean_transcript(text: str) -> str | None:
    """
    Normalise transcript. Returns None if annotation is too short or empty.
    """
    if not text or not text.strip():
        return None

    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)                        # collapse whitespace
    text = re.sub(r'[^\x00-\x7F]+', '', text)              # remove non-ASCII
    text = re.sub(r"[^a-z0-9\s\'\-\,\.\!\?]", '', text)   # keep useful chars
    text = text.strip()

    # Must have at least 2 words to be a valid lyric fragment
    if len(text.split()) < 2:
        return None

    return text

def parse_nus(root: str) -> list[dict]:
    """
    Parse NUS-48E dataset.
    Returns list of {audio_path, transcript, source, singer}.

    FIX from v1: NUS-48E is 2 levels deep (singer_dir/phrase.wav),
    NOT 3 levels. The extra song_dir loop in v1 caused 0 samples found.
    """
    root    = Path(root)
    samples = []

    if not root.exists():
        print(f"[SKIP] NUS-48E not found at {root}")
        return samples

    # Modified to look for WAV files directly in the root folder
    wav_files = sorted(root.glob("*.wav"))
    print(f"  Found {len(wav_files)} WAV files directly in {root}")

    for wav_file in wav_files:
        # Removed 'if "read" in wav_file.name.lower(): continue'
        # Assuming '01read_1.wav' are the files the user wants to process.

        txt_file = wav_file.with_suffix(".txt")
        if not txt_file.exists():
            print(f"    [WARN] No lyric file found for {wav_file.name} at {txt_file}")
            continue

        raw   = txt_file.read_text(encoding="utf-8", errors="ignore")
        trans = clean_transcript(" ".join(raw.splitlines()))
        if trans is None:
            continue

        samples.append({
            "audio_path": str(wav_file),
            "transcript": trans,
            "source":     "nus",
            "singer":     wav_file.stem,   # Use the file stem as the 'singer' identifier
        })

    print(f"[NUS-48E]  Found {len(samples)} valid samples "
          f"from {len({s['singer'] for s in samples})} 'singers' (derived from filenames).")
    return samples


def parse_jamendo(root: str) -> list[dict]:
    """
    Parse Jamendo lyrics dataset.
    Each long track is chunked into multiple 30s samples.

    FIX from v1:
    - Loads both .mp3 and .wav (some releases use .wav)
    - Each track becomes multiple samples via chunk_audio()
      instead of being trimmed to 30s (which lost ~90% of audio)
    """
    root       = Path(root)
    audio_dir  = root / "mp3"
    lyrics_dir = root / "lyrics"
    meta_path  = root / "metadata.csv"
    samples    = []

    if not root.exists():
        print(f"[SKIP] Jamendo not found at {root}")
        return samples

    print(f"  Jamendo audio_dir: {audio_dir}")
    print(f"  Jamendo lyrics_dir: {lyrics_dir}")

    # Optional: filter English-only tracks via metadata
    allowed_ids = None
    if meta_path.exists():
        meta     = pd.read_csv(meta_path)
        lang_col = next((c for c in meta.columns if "lang" in c.lower()), None)
        id_col   = next((c for c in meta.columns if "id"   in c.lower()), None)
        if lang_col and id_col:
            eng         = meta[meta[lang_col].str.lower() == "en"]
            allowed_ids = set(eng[id_col].astype(str).tolist())
            print(f"[Jamendo]  Metadata found — keeping {len(allowed_ids)} English tracks.")

    # ── FIX: glob both .mp3 and .wav ────────────────────────────
    audio_files = sorted(audio_dir.glob("*.mp3")) + sorted(audio_dir.glob("*.wav"))
    print(f"  Found {len(audio_files)} audio files in {audio_dir}")
    track_count, chunk_count, skip_count = 0, 0, 0

    for audio_file in tqdm(audio_files, desc="Parsing Jamendo"):
        track_id = audio_file.stem
        if allowed_ids is not None and track_id not in allowed_ids:
            continue

        txt_file = lyrics_dir / f"{track_id}.txt"
        if not txt_file.exists():
            print(f"    [WARN] No lyric file found for {audio_file.name} at {txt_file}")
            continue

        raw   = txt_file.read_text(encoding="utf-8", errors="ignore")
        trans = clean_transcript(" ".join(raw.splitlines()))
        if trans is None:
            skip_count += 1
            continue

        # Load once, chunk into multiple 30s samples
        waveform = load_and_resample(str(audio_file))
        if waveform is None:
            skip_count += 1
            continue

        duration = waveform.shape[0] / TARGET_SR
        if duration < MIN_DURATION:
            skip_count += 1
            continue

        # ── FIX: chunk instead of trim ───────────────────────────
        chunks = chunk_audio(waveform)
        for i, chunk in enumerate(chunks):
            samples.append({
                "audio_path": str(audio_file),  # same file — loaded lazily later
                "transcript": trans,
                "source":     "jamendo",
                "singer":     track_id,
                # Store chunk boundaries so lazy loader knows what to slice
                "chunk_start_sample": int(i * (MAX_DURATION - CHUNK_OVERLAP) * TARGET_SR),
                "chunk_end_sample":   int(i * (MAX_DURATION - CHUNK_OVERLAP) * TARGET_SR) + len(chunk),
            })
            chunk_count += 1
        track_count += 1

    print(f"[Jamendo]  Processed {track_count} tracks → {chunk_count} chunks "
          f"({skip_count} tracks skipped).")
    return samples

def split_dataset(all_samples: list[dict]) -> dict[str, list]:
    """
    Stratified 70/15/15 split.
    - NUS-48E  : split at SINGER level (prevents data leakage)
    - Jamendo  : split at track level  (each track is independent)
    """
    by_source = {}
    for s in all_samples:
        by_source.setdefault(s["source"], []).append(s)

    train, val, test = [], [], []

    # ── NUS-48E: singer-level split ──────────────────────────────
    nus_samples = by_source.get("nus", [])
    if nus_samples:
        singers = sorted({s["singer"] for s in nus_samples})
        random.shuffle(singers)
        n_s          = len(singers)
        n_train_s    = max(1, int(n_s * TRAIN_RATIO))
        n_val_s      = max(1, int(n_s * VAL_RATIO))

        train_singers = set(singers[:n_train_s])
        val_singers   = set(singers[n_train_s : n_train_s + n_val_s])
        test_singers  = set(singers[n_train_s + n_val_s :])

        nus_train = [s for s in nus_samples if s["singer"] in train_singers]
        nus_val   = [s for s in nus_samples if s["singer"] in val_singers]
        nus_test  = [s for s in nus_samples if s["singer"] in test_singers]

        train += nus_train
        val   += nus_val
        test  += nus_test

        print(f"  [nus]     singers total={n_s} | "
              f"train={len(nus_train)} ({len(train_singers)} singers) | "
              f"val={len(nus_val)} ({len(val_singers)} singers) | "
              f"test={len(nus_test)} ({len(test_singers)} singers)")

    # ── Jamendo: track-level split ───────────────────────────────
    jam_samples = by_source.get("jamendo", [])
    if jam_samples:
        # Group chunks by track, split tracks, then flatten
        tracks = sorted({s["singer"] for s in jam_samples})
        random.shuffle(tracks)
        n_t       = len(tracks)
        n_train_t = max(1, int(n_t * TRAIN_RATIO))
        n_val_t   = max(1, int(n_t * VAL_RATIO))

        train_tracks = set(tracks[:n_train_t])
        val_tracks   = set(tracks[n_train_t : n_train_t + n_val_t])

        jam_train = [s for s in jam_samples if s["singer"] in train_tracks]
        jam_val   = [s for s in jam_samples if s["singer"] in val_tracks]
        jam_test  = [s for s in jam_samples if s["singer"] not in train_tracks | val_tracks]

        train += jam_train
        val   += jam_val
        test  += jam_test

        print(f"  [jamendo] tracks total={n_t} | "
              f"train={len(jam_train)} chunks | "
              f"val={len(jam_val)} chunks | "
              f"test={len(jam_test)} chunks")

    # Final shuffle within each split
    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)

    print(f"\n  Total — train: {len(train)} | val: {len(val)} | test: {len(test)}")
    return {"train": train, "val": val, "test": test}

class LyricsDataset(TorchDataset):
    """
    Lazy-loading dataset for Whisper fine-tuning.

    Each sample dict must contain:
      - audio_path           : path to audio file
      - transcript           : cleaned text string
      - source               : "nus" or "jamendo"
      - chunk_start_sample   : (Jamendo only) start sample index
      - chunk_end_sample     : (Jamendo only) end sample index

    __getitem__ loads + processes on demand — no pre-computed spectrograms.
    """

    def __init__(self, samples: list[dict]):
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        s = self.samples[idx]

        # ── 1. Load audio ─────────────────────────────────────────
        waveform = load_and_resample(s["audio_path"])
        if waveform is None:
            # Return a silent fallback rather than crashing the DataLoader
            waveform = torch.zeros(int(MAX_DURATION * TARGET_SR))

        # ── 2. Extract the right chunk ────────────────────────────
        if s["source"] == "jamendo" and "chunk_start_sample" in s:
            waveform = waveform[s["chunk_start_sample"] : s["chunk_end_sample"]]
        else:
            # NUS-48E phrases are short — just trim to 30s max
            waveform = pad_or_trim(waveform)

        # ── 3. Extract log-mel spectrogram ────────────────────────
        features = processor.feature_extractor(
            waveform.numpy(),
            sampling_rate=TARGET_SR,
            return_tensors="pt",
        ).input_features[0]  # [80, 3000]

        # ── 4. Tokenize transcript ────────────────────────────────
        # FIX: add_special_tokens=False removes BOS token from labels.
        # Including BOS in labels causes a 1-position misalignment with
        # decoder inputs → loss computed on wrong target tokens.
        labels = processor.tokenizer(
            s["transcript"],
            add_special_tokens=False,
        ).input_ids

        return {
            "input_features": features,                          # [80, 3000] float32
            "labels":         torch.tensor(labels, dtype=torch.long),  # variable length
            "transcript":     s["transcript"],                   # for WER eval later
            "source":         s["source"],
        }


def collate_fn(batch: list[dict]) -> dict:
    """
    Custom collate for variable-length labels.

    - input_features: stacked to [B, 80, 3000] (all same shape)
    - labels: padded to [B, max_label_len] with -100
              CrossEntropyLoss ignores -100 automatically
    - transcript: kept as a list of strings
    """
    input_features = torch.stack([b["input_features"] for b in batch])  # [B, 80, 3000]

    label_lengths  = [len(b["labels"]) for b in batch]
    max_len        = max(label_lengths)

    # -100 is the standard ignore index for CrossEntropyLoss
    padded_labels  = torch.full(
        (len(batch), max_len), fill_value=-100, dtype=torch.long
    )
    for i, b in enumerate(batch):
        padded_labels[i, : len(b["labels"])] = b["labels"]

    return {
        "input_features": input_features,
        "labels":         padded_labels,
        "transcript":     [b["transcript"] for b in batch],
        "source":         [b["source"]     for b in batch],
    }

print("=" * 55)
print("Step 2a: Parsing raw datasets from disk...")
print("=" * 55)

raw = []
raw += parse_nus(NUS_ROOT)
raw += parse_jamendo(JAMENDO_ROOT)

if not raw:
    print("\n[ERROR] No samples found. Check NUS_ROOT and JAMENDO_ROOT paths.")
else:
    print(f"\nTotal raw samples (after parsing): {len(raw)}")
    print(f"  NUS-48E  : {sum(1 for s in raw if s['source'] == 'nus')}")
    print(f"  Jamendo  : {sum(1 for s in raw if s['source'] == 'jamendo')}")

    print()
    print("=" * 55)
    print("Step 2e: Splitting dataset (singer/track-level stratified)...")
    print("=" * 55)

    splits = split_dataset(raw)

    print()
    print("=" * 55)
    print("Wrapping splits in LyricsDataset...")
    print("=" * 55)

    train_dataset = LyricsDataset(splits["train"])
    val_dataset   = LyricsDataset(splits["val"])
    test_dataset  = LyricsDataset(splits["test"])

    print(f"  train_dataset : {len(train_dataset)} samples")
    print(f"  val_dataset   : {len(val_dataset)} samples")
    print(f"  test_dataset  : {len(test_dataset)} samples")

# ── Sanity Check 1: Single sample shape ──────────────────────
print("── Sanity Check 1: Single sample from train_dataset ──")
sample = train_dataset[0]
print(f"  input_features shape : {sample['input_features'].shape}")
print(f"  labels length        : {len(sample['labels'])}")
print(f"  labels dtype         : {sample['labels'].dtype}")
print(f"  transcript (first 60): {sample['transcript'][:60]}")
print(f"  source               : {sample['source']}")

assert sample['input_features'].shape == torch.Size([80, 3000]), \
    f"Expected [80, 3000], got {sample['input_features'].shape}"
assert sample['labels'][0] != processor.tokenizer.bos_token_id, \
    "BOS token found in labels — add_special_tokens=False not applied correctly"
print("  ✓ Shape and BOS check passed")

print()

# ── Sanity Check 2: DataLoader batch ─────────────────────────
print("── Sanity Check 2: DataLoader batch (batch_size=4) ──")
loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,   # set to 2-4 in training for speed
)
batch = next(iter(loader))

print(f"  input_features shape : {batch['input_features'].shape}")
print(f"  labels shape         : {batch['labels'].shape}")
print(f"  label padding (-100) : {(batch['labels'] == -100).sum().item()} positions padded")
print(f"  sources in batch     : {batch['source']}")
print(f"  transcript[0][:60]   : {batch['transcript'][0][:60]}")

assert batch['input_features'].shape[1:] == torch.Size([80, 3000])
assert batch['labels'].shape[0] == 4
print("  ✓ Batch shapes correct")

print()
print("Phase 2 complete ✓  Ready for Phase 3 (LoRA Injection).")

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, TaskType

# ── Import from Phase 1 ───────────────────────────────────────
# These must be in memory from running step1_setup


print(f"Device        : {DEVICE}")
print(f"Model class   : {model.__class__.__name__}")
print(f"Model loaded  : ✓")

# ─────────────────────────────────────────────────────────────
# Step 3a: LoRA hyperparameters
# ─────────────────────────────────────────────────────────────

# LoRA config — will be assigned to target_modules below
lora_config = LoraConfig(
    # Task type: Whisper is encoder-decoder (seq2seq)
    task_type=TaskType.SEQ_2_SEQ_LM,

    # LoRA rank & scaling
    r=8,                      # rank of LoRA matrices
    lora_alpha=16,            # scaling factor (typically 2x r)

    # Regularization
    lora_dropout=0.05,        # 5% dropout on LoRA weights

    # Target modules — set below in Step 3b
    target_modules=None,      # will be set after we identify module names

    # Other options
    bias="none",             # don't add bias to LoRA matrices
    modules_to_save=None,     # no additional modules to fine-tune fully
)

print("Step 3a: LoRA config created (target_modules not yet assigned)")
print(f"  r (rank)               : {lora_config.r}")
print(f"  lora_alpha             : {lora_config.lora_alpha}")
print(f"  lora_dropout           : {lora_config.lora_dropout}")
print(f"  task_type              : {lora_config.task_type}")

# ─────────────────────────────────────────────────────────────
# Step 3b: Find target modules in the decoder
# ─────────────────────────────────────────────────────────────

print("Step 3b: Identifying target modules in decoder...\n")

# List all named modules to understand the structure
# We're looking for decoder attention layers with q_proj, v_proj
model_modules = dict(model.named_modules())
decoder_attention_modules = {}

for name, module in model_modules.items():
    # Search for decoder attention layers
    if "decoder" in name and ("q_proj" in name or "v_proj" in name):
        decoder_attention_modules[name] = module.__class__.__name__

print(f"Found {len(decoder_attention_modules)} decoder attention projection layers:")
for name in sorted(decoder_attention_modules.keys()):
    print(f"  {name}")

# Extract unique module types to target
# e.g., all instances of q_proj and v_proj in the decoder
target_modules = []
if decoder_attention_modules:
    # Get the last component of module names (e.g., "q_proj", "v_proj")
    module_suffixes = set()
    for name in decoder_attention_modules.keys():
        parts = name.split(".")
        if parts[-1] in ["q_proj", "v_proj"]:
            module_suffixes.add(parts[-1])
    target_modules = sorted(list(module_suffixes))

print(f"\nTarget modules to adapt: {target_modules}")
print(f"  → LoRA adapters will be attached to all {target_modules} layers in the decoder")

# ─────────────────────────────────────────────────────────────
# Step 3c: Update config with target modules and inject LoRA
# ─────────────────────────────────────────────────────────────

# Update the config with identified target modules
lora_config.target_modules = target_modules

print("Step 3c: Injecting LoRA adapters into model...\n")

# Wrap the model with PEFT
# This freezes the base model and adds LoRA matrices to target modules
model = get_peft_model(model, lora_config)

print("LoRA injection complete ✓")
print(f"  Model class: {model.__class__.__name__}")
print(f"  Base model: {model.base_model.__class__.__name__}")

print("=" * 60)
print("PARAMETER ANALYSIS (after LoRA injection)")
print("=" * 60)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nTotal parameters       : {total_params / 1e6:,.1f}M")
print(f"  └─ Trainable (LoRA)   : {trainable_params / 1e6:,.1f}M ({100 * trainable_params / total_params:.2f}%)")
print(f"  └─ Frozen (base)      : {frozen_params / 1e6:,.1f}M ({100 * frozen_params / total_params:.2f}%)")

print()
print("LoRA module breakdown:")

# Count LoRA-specific parameters
lora_params = 0
lora_module_count = 0
for name, param in model.named_parameters():
    if "lora" in name and param.requires_grad:
        lora_params += param.numel()
        if "lora_B" in name:
            lora_module_count += 1

print(f"  LoRA parameters       : {lora_params / 1e6:,.1f}M")
print(f"  LoRA adapter pairs    : {lora_module_count} (q_proj + v_proj per decoder layer)")

print()
print("Trainable modules:")
trainable_modules = set()
for name, param in model.named_parameters():
    if param.requires_grad:
        module_name = ".".join(name.split(".")[:-1])  # remove param name
        trainable_modules.add(module_name)

# Show a few examples
for name in sorted(list(trainable_modules))[:10]:
    print(f"  ✓ {name}")

if len(trainable_modules) > 10:
    print(f"  ... and {len(trainable_modules) - 10} more")

print("\n" + "=" * 60)
print("SANITY CHECKS")
print("=" * 60)

# ── Check 1: Base model is frozen ────────────────────────────
print("\nCheck 1: Base model parameters are frozen")
base_frozen = True
for name, param in model.base_model.named_parameters():
    if param.requires_grad:
        base_frozen = False
        print(f"  ✗ Found trainable base param: {name}")
        break

if base_frozen:
    print("  ✓ All base model parameters frozen")

# ── Check 2: LoRA adapters have gradients ───────────────────
print("\nCheck 2: LoRA adapters are trainable")
lora_trainable = False
for name, param in model.named_parameters():
    if "lora" in name and param.requires_grad:
        lora_trainable = True
        break

if lora_trainable:
    print("  ✓ LoRA adapters are trainable")
else:
    print("  ✗ No trainable LoRA adapters found!")

# ── Check 3: Model can still generate ────────────────────────
print("\nCheck 3: Model forward pass (sanity check)")
try:
    # Create a dummy batch
    dummy_input_features = torch.randn(1, 80, 3000).to(DEVICE)
    dummy_labels = torch.tensor([[50258, 50259, 1234, 5678, 50256]]).to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_features=dummy_input_features,
            labels=dummy_labels,
        )

    print(f"  ✓ Model forward pass succeeded")
    print(f"  ✓ Loss shape: {outputs.loss.shape}")
    print(f"  ✓ Logits shape: {outputs.logits.shape}")
except Exception as e:
    print(f"  ✗ Forward pass failed: {e}")

# ── Check 4: Verify no quantization issues ───────────────────
print("\nCheck 4: Model is ready for training")
if hasattr(model, 'base_model'):
    print(f"  ✓ PEFT wrapper in place")
else:
    print(f"  ✗ PEFT wrapper missing")

print()
print("Phase 3 complete ✓  Ready for Phase 4 (Training).")

In [ ]:
# -----------------------------------------------------------------------------
# PATCH -- Fix ValueError: provide either input_features or encoder_outputs
#
# ROOT CAUSE (full picture after 3 errors):
#
#   PEFT wraps model.generate() and passes kwargs to base_model.generate()
#   BUT it strips any kwarg not in its whitelist before passing through.
#
#   model.generate(input_features=...)  -> PEFT strips input_features -> Whisper gets nothing -> crash
#   model.generate(inputs=...)          -> PEFT strips inputs          -> Whisper gets nothing -> crash
#
# DEFINITIVE FIX:
#   Call generate directly on the UNWRAPPED base model:
#       model.base_model.model.generate(input_features=input_features, ...)
#
#   The base model is pure Whisper with LoRA weights merged in-place,
#   so it accepts input_features= directly with no PEFT interference.
# -----------------------------------------------------------------------------

import torch, json, os
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
from jiwer import wer, cer

MAX_LABEL_TOKENS = 444
BATCH_SIZE       = 8
SAVE_DIR         = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# -- Verify the correct generate target ---------------------------------------
print("Model class hierarchy:")
print(f"  model              : {model.__class__.__name__}")
print(f"  model.base_model   : {model.base_model.__class__.__name__}")
print(f"  model.base_model.model : {model.base_model.model.__class__.__name__}")
print()
print("  -> Will call generate on: model.base_model.model")
print("     (pure WhisperForConditionalGeneration, no PEFT wrapper)")


def collate_fn_fixed(batch):
    """Label-truncating collator -- prevents >448 token error."""
    input_features = torch.stack([b["input_features"] for b in batch])
    truncated      = [b["labels"][:MAX_LABEL_TOKENS] for b in batch]
    max_len        = max(len(l) for l in truncated)
    padded         = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, l in enumerate(truncated):
        padded[i, :len(l)] = l
    return {
        "input_features": input_features,
        "labels":         padded,
        "transcript":     [b["transcript"] for b in batch],
        "source":         [b["source"]     for b in batch],
    }


def evaluate_model(model, dataset, split_name="val", n_samples=None):
    """
    DEFINITIVE fixed evaluate_model.

    generate() is called on model.base_model.model -- the raw
    WhisperForConditionalGeneration -- which accepts input_features=
    directly without PEFT interference.

    model.forward() (for loss) still uses the PEFT-wrapped model
    so LoRA weights are active during loss computation.
    """
    if n_samples is not None:
        dataset = Subset(dataset, range(min(n_samples, len(dataset))))

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_fixed,
    )

    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_refs   = []

    # Unwrap once outside the loop for efficiency
    whisper_model = model.base_model.model   # raw WhisperForConditionalGeneration

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Evaluating {split_name}", leave=False):

            input_features = batch["input_features"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            refs           = batch["transcript"]

            # Loss: use PEFT-wrapped model (LoRA active)
            outputs = model(
                input_features=input_features,
                labels=labels,
            )
            total_loss += outputs.loss.item()

            # Generate: use unwrapped Whisper directly
            # input_features= works here because no PEFT layer intercepts it
            generated_ids = whisper_model.generate(
                input_features=input_features,
                num_beams=1,
                max_new_tokens=128,
            )

            preds = processor.tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
            )

            all_preds.extend(preds)
            all_refs.extend(refs)

    avg_loss = total_loss / len(loader)

    # Filter empty pairs
    pairs = [
        (r, p) for r, p in zip(all_refs, all_preds)
        if r.strip() and p.strip()
    ]
    if pairs:
        avg_wer = wer([p[0] for p in pairs], [p[1] for p in pairs])
        avg_cer = cer([p[0] for p in pairs], [p[1] for p in pairs])
    else:
        avg_wer = 1.0
        avg_cer = 1.0
        print(f"  [WARN] All predictions empty in {split_name}")

    model.train()
    return avg_loss, avg_wer, avg_cer


print("\n  evaluate_model redefined -- generate() on base_model.model")
print("-" * 60)

# -----------------------------------------------------------------------------
# Zero-shot baseline (val set -- for reference before training)
# -----------------------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPERIMENT 1: Zero-shot baseline")
print("=" * 60)

zs_loss, zs_wer_score, zs_cer_score = evaluate_model(
    model, val_dataset, "zero-shot", n_samples=50
)

print(f"\n  Zero-shot on 50 val samples:")
print(f"  Loss : {zs_loss:.4f}")
print(f"  WER  : {zs_wer_score * 100:.2f}%  <- baseline to beat")
print(f"  CER  : {zs_cer_score * 100:.2f}%  <- baseline to beat")

results_log = {
    "zero_shot": {"loss": zs_loss, "wer": zs_wer_score, "cer": zs_cer_score}
}
with open(f"{SAVE_DIR}/results_log.json", "w") as f:
    json.dump(results_log, f, indent=2)

print(f"  Saved to {SAVE_DIR}/results_log.json")

# -----------------------------------------------------------------------------
# EXPERIMENT 2: LoRA decoder-only fine-tuning
# -----------------------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPERIMENT 2: LoRA decoder-only -- Training")
print("=" * 60)

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Training hyperparameters
EPOCHS       = 10     # increase to 20 if val WER is still falling at epoch 10
LR           = 1e-4   # standard starting point for LoRA fine-tuning
WARMUP_STEPS = 50     # linear warmup over first 50 steps
GRAD_CLIP    = 1.0    # gradient clipping norm

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn_fixed,
    num_workers=0,
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
)
total_steps = EPOCHS * len(train_loader)
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

print(f"  Trainable parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}M")
print(f"  Epochs               : {EPOCHS}")
print(f"  Learning rate        : {LR}")
print(f"  Warmup steps         : {WARMUP_STEPS}")
print(f"  Steps per epoch      : {len(train_loader)}")
print(f"  Total steps          : {total_steps}")
print()

# Training loop
best_val_wer = float("inf")
training_log = []

model.train()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    n_batches  = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True):
        input_features = batch["input_features"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        outputs = model(input_features=input_features, labels=labels)
        loss    = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            GRAD_CLIP,
        )
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        n_batches  += 1

    avg_train_loss = epoch_loss / n_batches

    # Validate after every epoch
    val_loss, val_wer_score, val_cer_score = evaluate_model(
        model, val_dataset, "val", n_samples=50
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_WER={val_wer_score:.2%} | "
        f"val_CER={val_cer_score:.2%}"
    )

    training_log.append({
        "epoch":      epoch,
        "train_loss": avg_train_loss,
        "val_loss":   val_loss,
        "val_wer":    val_wer_score,
        "val_cer":    val_cer_score,
    })

    # Save best checkpoint (lowest val WER)
    if val_wer_score < best_val_wer:
        best_val_wer = val_wer_score
        model.save_pretrained("./checkpoints/lora_decoder")
        processor.save_pretrained("./checkpoints/lora_decoder")
        print(f"  Best checkpoint saved (val WER: {best_val_wer:.2%})")

# Persist training log
results_log["lora_decoder"] = {
    "best_val_wer": best_val_wer,
    "training_log": training_log,
}
with open(f"{SAVE_DIR}/results_log.json", "w") as f:
    json.dump(results_log, f, indent=2)

print(f"\nTraining complete.  Best val WER: {best_val_wer:.2%}")
print(f"Checkpoint: ./checkpoints/lora_decoder")
print("\nPhase 4 complete   Ready for Phase 5 (Evaluation).")

# -----------------------------------------------------------------------------
# PHASE 5 -- EVALUATION
# |-- Step 5a: Run inference on held-out TEST set for all 3 experiments
# |-- Step 5b: Calculate WER and CER using Jiwer
# |-- Step 5c: Log VRAM usage and inference latency per experiment
# +-- Step 5d: Compare results in a printed table
#
# DEPENDS ON (must already be in memory from earlier phases):
#   - model          : PEFT-wrapped Whisper (after LoRA training)
#   - processor      : WhisperProcessor
#   - test_dataset   : held-out test split (LyricsDataset)
#   - results_log    : dict populated in Phase 4 with zero_shot + lora_decoder keys
#   - DEVICE, SAVE_DIR, BATCH_SIZE, MAX_LABEL_TOKENS, collate_fn_fixed
#
# NOTE: This cell runs AFTER Phase 4 training is complete.
#       It reloads each checkpoint from disk so results are reproducible.
# -----------------------------------------------------------------------------

import os
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from jiwer import wer, cer
from torch.utils.data import DataLoader, Subset
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

print("=" * 60)
print("PHASE 5 -- EVALUATION")
print("=" * 60)

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------
SAVE_DIR         = "./checkpoints"        # same as Phase 4
N_TEST_SAMPLES   = None                   # None = use full test set; set e.g. 100 for quick run
BEAM_SIZE        = 1                      # keep at 1 for fair latency comparison
MAX_NEW_TOKENS   = 128

# Checkpoint paths saved during Phase 4
CKPT_LORA_DECODER   = f"{SAVE_DIR}/lora_decoder"
CKPT_LORA_BOTH      = f"{SAVE_DIR}/lora_both"   # only exists if Experiment 3 was run

os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# HELPER: VRAM snapshot
# -----------------------------------------------------------------------------
def get_vram_mb() -> float:
    """Return current GPU memory allocated in MB. Returns 0.0 on CPU."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 ** 2
    return 0.0

def reset_vram_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def get_peak_vram_mb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024 ** 2
    return 0.0


# -----------------------------------------------------------------------------
# CORE EVALUATION FUNCTION
# -----------------------------------------------------------------------------
# Uses the same collate_fn_fixed and unwrap trick from Phase 4.
# Additionally records:
#   - per-batch latency  -> average latency per sample (ms)
#   - peak VRAM          -> max GPU memory during inference

def evaluate_on_test(model, dataset, experiment_name: str, n_samples=None):
    """
    Run inference on test set and return metrics dict:
    {
        "wer":              float,
        "cer":              float,
        "loss":             float,
        "avg_latency_ms":   float,   # per sample
        "peak_vram_mb":     float,
        "n_samples":        int,
        "predictions":      list[str],
        "references":       list[str],
    }
    """
    print(f"\n{'-' * 60}")
    print(f"  Evaluating: {experiment_name}")
    print(f"{'-' * 60}")

    if n_samples is not None:
        dataset = Subset(dataset, range(min(n_samples, len(dataset))))

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_fixed,
    )

    model.eval()
    total_loss   = 0.0
    all_preds    = []
    all_refs     = []
    latencies_ms = []

    # Unwrap PEFT once -- generate() on raw Whisper avoids PEFT kwarg stripping
    # If model is not a PEFT model (e.g. pure Whisper), use model directly
    if hasattr(model, "base_model"):
        whisper_model = model.base_model.model
    else:
        whisper_model = model

    reset_vram_peak()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  {experiment_name}", leave=True):

            input_features = batch["input_features"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            refs           = batch["transcript"]
            batch_size     = input_features.shape[0]

            # Loss (PEFT-wrapped forward, LoRA active)
            outputs    = model(input_features=input_features, labels=labels)
            total_loss += outputs.loss.item()

            # Timed generation
            t_start       = time.perf_counter()
            generated_ids = whisper_model.generate(
                input_features=input_features,
                num_beams=BEAM_SIZE,
                max_new_tokens=MAX_NEW_TOKENS,
            )
            t_end = time.perf_counter()

            # Latency per sample in ms
            elapsed_ms = (t_end - t_start) * 1000
            latencies_ms.extend([elapsed_ms / batch_size] * batch_size)

            preds = processor.tokenizer.batch_decode(
                generated_ids, skip_special_tokens=True
            )

            all_preds.extend(preds)
            all_refs.extend(refs)

    # Aggregate metrics
    avg_loss       = total_loss / len(loader)
    avg_latency_ms = float(np.mean(latencies_ms))
    peak_vram_mb   = get_peak_vram_mb()

    # Filter empty pairs before WER/CER
    pairs = [
        (r.strip(), p.strip())
        for r, p in zip(all_refs, all_preds)
        if r.strip() and p.strip()
    ]

    if pairs:
        refs_clean  = [p[0] for p in pairs]
        preds_clean = [p[1] for p in pairs]
        avg_wer     = wer(refs_clean, preds_clean)
        avg_cer     = cer(refs_clean, preds_clean)
    else:
        avg_wer = 1.0
        avg_cer = 1.0
        print(f"  [WARN] All predictions were empty for {experiment_name}")

    # Print summary
    print(f"\n  Results for [{experiment_name}]")
    print(f"    Samples evaluated  : {len(all_refs)}")
    print(f"    Loss               : {avg_loss:.4f}")
    print(f"    WER                : {avg_wer * 100:.2f}%")
    print(f"    CER                : {avg_cer * 100:.2f}%")
    print(f"    Avg latency/sample : {avg_latency_ms:.1f} ms")
    print(f"    Peak VRAM          : {peak_vram_mb:.1f} MB  {'(CPU -- N/A)' if DEVICE == 'cpu' else ''}")

    # Show 3 sample predictions
    print(f"\n  Sample predictions:")
    for i in range(min(3, len(all_refs))):
        print(f"    REF  : {all_refs[i][:80]}")
        print(f"    PRED : {all_preds[i][:80]}")
        print()

    model.train()

    return {
        "wer":            avg_wer,
        "cer":            avg_cer,
        "loss":           avg_loss,
        "avg_latency_ms": avg_latency_ms,
        "peak_vram_mb":   peak_vram_mb,
        "n_samples":      len(all_refs),
        "predictions":    all_preds,
        "references":     all_refs,
    }


# -----------------------------------------------------------------------------
# HELPER: load a saved LoRA checkpoint back onto base model
# -----------------------------------------------------------------------------
def load_lora_checkpoint(checkpoint_path: str):
    """
    Load a LoRA checkpoint saved with model.save_pretrained().
    Returns a PEFT-wrapped model on DEVICE.
    """
    print(f"\nLoading checkpoint: {checkpoint_path}")

    # Reload clean base model
    base = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    base.config.forced_decoder_ids = None
    base.config.suppress_tokens    = []

    # Wrap with saved LoRA weights
    peft_model = PeftModel.from_pretrained(base, checkpoint_path)
    peft_model = peft_model.to(DEVICE)
    peft_model.eval()

    print(f"  Checkpoint loaded from {checkpoint_path}")
    return peft_model


# -----------------------------------------------------------------------------
# STEP 5a + 5b + 5c -- Run all experiments on TEST set
# -----------------------------------------------------------------------------

test_results = {}

# -- Experiment 1: Zero-shot --------------------------------------------------
# Reload a clean base Whisper model with no LoRA adapters.
# This gives a proper zero-shot baseline that is independent of
# whatever LoRA weights are currently in memory from Phase 4.
print("\n" + "=" * 60)
print("EXPERIMENT 1: Zero-shot baseline (TEST SET)")
print("=" * 60)

print("Loading clean base Whisper-small for zero-shot baseline...")
zs_base = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
zs_base.config.forced_decoder_ids = None
zs_base.config.suppress_tokens    = []
zs_base = zs_base.to(DEVICE)
zs_base.eval()
print("  Clean base model loaded")

test_results["zero_shot"] = evaluate_on_test(
    zs_base, test_dataset,
    experiment_name="Zero-shot",
    n_samples=N_TEST_SAMPLES,
)
del zs_base
torch.cuda.empty_cache() if DEVICE == "cuda" else None

# -- Experiment 2: LoRA decoder-only ------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 2: LoRA -- Decoder only (TEST SET)")
print("=" * 60)

if os.path.exists(CKPT_LORA_DECODER):
    lora_decoder_model = load_lora_checkpoint(CKPT_LORA_DECODER)
    test_results["lora_decoder"] = evaluate_on_test(
        lora_decoder_model, test_dataset,
        experiment_name="LoRA Decoder-only",
        n_samples=N_TEST_SAMPLES,
    )
    del lora_decoder_model          # free VRAM before next experiment
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
else:
    print(f"  [SKIP] Checkpoint not found at {CKPT_LORA_DECODER}")
    print(f"         Run Phase 4 Experiment 2 first and save with model.save_pretrained()")

# -- Experiment 3: LoRA encoder + decoder (optional) --------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 3: LoRA -- Encoder + Decoder (TEST SET)")
print("=" * 60)

if os.path.exists(CKPT_LORA_BOTH):
    lora_both_model = load_lora_checkpoint(CKPT_LORA_BOTH)
    test_results["lora_both"] = evaluate_on_test(
        lora_both_model, test_dataset,
        experiment_name="LoRA Encoder+Decoder",
        n_samples=N_TEST_SAMPLES,
    )
    del lora_both_model
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
else:
    print(f"  [SKIP] Checkpoint not found at {CKPT_LORA_BOTH}")
    print(f"         This is optional -- only needed for Experiment 3.")


# -----------------------------------------------------------------------------
# STEP 5d -- COMPARISON TABLE
# -----------------------------------------------------------------------------

print("\n\n" + "=" * 70)
print("STEP 5d -- RESULTS COMPARISON TABLE")
print("=" * 70)

# Also pull zero-shot val numbers from Phase 4 results_log (for reference)
try:
    with open(f"{SAVE_DIR}/results_log.json") as f:
        phase4_log = json.load(f)
except FileNotFoundError:
    phase4_log = {}

# Header
header = f"{'Experiment':<28} {'WER':>8} {'CER':>8} {'Loss':>8} {'Latency(ms)':>13} {'VRAM(MB)':>10}"
print(header)
print("-" * 70)

experiment_labels = {
    "zero_shot":    "1. Zero-shot (baseline)",
    "lora_decoder": "2. LoRA Decoder-only",
    "lora_both":    "3. LoRA Encoder+Decoder",
}

baseline_wer = test_results.get("zero_shot", {}).get("wer", None)

for key, label in experiment_labels.items():
    if key not in test_results:
        print(f"  {label:<26} {'--':>8} {'--':>8} {'--':>8} {'--':>13} {'--':>10}  (not run)")
        continue

    r       = test_results[key]
    wer_pct = r["wer"] * 100
    cer_pct = r["cer"] * 100

    # Show relative WER reduction vs baseline
    rel_improvement = ""
    if baseline_wer and key != "zero_shot":
        rel_wer_reduction = (baseline_wer - r["wer"]) / baseline_wer * 100
        direction         = "down" if rel_wer_reduction > 0 else "up"
        rel_improvement   = f"  ({direction} {abs(rel_wer_reduction):.1f}% WER)"

    print(
        f"  {label:<26} "
        f"{wer_pct:>7.2f}% "
        f"{cer_pct:>7.2f}% "
        f"{r['loss']:>8.4f} "
        f"{r['avg_latency_ms']:>11.1f}ms "
        f"{r['peak_vram_mb']:>8.1f}MB"
        f"{rel_improvement}"
    )

print("-" * 70)

# Project target: >15% relative WER reduction
if baseline_wer:
    target_wer = baseline_wer * 0.85
    print(f"\n  Target WER (>15% relative reduction from {baseline_wer*100:.2f}%): {target_wer*100:.2f}%")
    for key in ["lora_decoder", "lora_both"]:
        if key in test_results:
            achieved = test_results[key]["wer"] <= target_wer
            status   = "TARGET MET" if achieved else "below target"
            print(f"  {experiment_labels[key]}: {status}")


# -----------------------------------------------------------------------------
# SAVE FULL RESULTS TO DISK
# -----------------------------------------------------------------------------

# Save metrics (no predictions -- too large)
save_data = {}
for key, r in test_results.items():
    save_data[key] = {
        "wer":            r["wer"],
        "cer":            r["cer"],
        "loss":           r["loss"],
        "avg_latency_ms": r["avg_latency_ms"],
        "peak_vram_mb":   r["peak_vram_mb"],
        "n_samples":      r["n_samples"],
    }

results_path = f"{SAVE_DIR}/phase5_test_results.json"
with open(results_path, "w") as f:
    json.dump(save_data, f, indent=2)

print(f"\n  Metrics saved to {results_path}")

# Save predictions for each experiment (useful for error analysis in report)
for key, r in test_results.items():
    preds_path = f"{SAVE_DIR}/predictions_{key}.json"
    with open(preds_path, "w") as f:
        json.dump(
            [{"ref": ref, "pred": pred}
             for ref, pred in zip(r["references"], r["predictions"])],
            f, indent=2
        )
    print(f"  Predictions saved to {preds_path}")

# -----------------------------------------------------------------------------
# SAVE evaluation_results.json -- required by Phase 6
# -----------------------------------------------------------------------------
# Phase 6 reads this file to load E2 predictions for post-correction.
# Structure mirrors what Phase 6 expects at /content/evaluation_results.json.

_key_map = {
    "zero_shot":    "experiment_1",
    "lora_decoder": "experiment_2",
    "lora_both":    "experiment_3",
}

eval_report = {
    "test_set_size": len(test_dataset),
    "device":        DEVICE,
    "results": {
        _key_map[key]: {
            "wer":          r["wer"],
            "cer":          r["cer"],
            "latency_ms":   r["avg_latency_ms"],
            "peak_vram_mb": r["peak_vram_mb"],
        }
        for key, r in test_results.items()
    },
    "predictions": {
        "experiment_2": {
            "predictions": test_results["lora_decoder"]["predictions"],
            "references":  test_results["lora_decoder"]["references"],
        }
    } if "lora_decoder" in test_results else {},
}

eval_report_path = "/content/evaluation_results.json"
with open(eval_report_path, "w") as f:
    json.dump(eval_report, f, indent=2)

print(f"  Phase 6 input saved to {eval_report_path}")

print("\nPhase 5 complete   Ready for Phase 6 (Post Correction) or Phase 7 (Report).")
